# GridFM: a neural solver for AC Optimal Power Flow


## Setup

**Colab:** Runtime → Change runtime type → **T4 GPU**, then run the next cell. If a previous install crashed: Runtime → Disconnect and delete runtime.

**Local:** open this notebook from a clone of `gridfm-graphkit` (kernel with `gridfm-graphkit` installed). The next cell skips the clone/pip path.

As the install progresses, you can start reading the introduction.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    REPO = Path("/content/gridfm-graphkit")
    GIT_BRANCH, GIT_URL = "lfe-tutorial", "https://github.com/gridfm/gridfm-graphkit.git"

    def _run(cmd):
        print("+", " ".join(map(str, cmd)))
        subprocess.check_call(cmd)

    marker = REPO / "examples" / "notebooks" / "tutorial_opf_helpers.py"
    if not marker.is_file():
        _run(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_URL, str(REPO)])
    else:
        _run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", GIT_BRANCH])
        _run(["git", "-C", str(REPO), "reset", "--hard", f"origin/{GIT_BRANCH}"])
    _run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(REPO)])
    keep = Path("/tmp/colab-keep.txt")
    keep.write_text("torch\ntorchaudio\ntorchvision\nnumpy\npandas\n")
    need = [
        spec
        for module, spec in (
            ("mlflow", "mlflow"),
            ("lightning", "lightning"),
            ("torch_geometric", "torch-geometric"),
            ("huggingface_hub", "huggingface_hub"),
        )
        if importlib.util.find_spec(module) is None
    ]
    if need:
        _run([sys.executable, "-m", "pip", "install", "-q", "-c", str(keep), *need])
    if importlib.util.find_spec("torch_scatter") is None:
        import torch

        _run([
            sys.executable, "-m", "pip", "install", "-q", "-c", str(keep),
            "torch-scatter", "-f", f"https://data.pyg.org/whl/torch-{torch.__version__}.html",
        ])
    os.chdir(REPO)
    DATA = Path("/content/data")
    LOG = Path("/content/mlruns")
    if Path("/content/drive/MyDrive").exists():
        LOG = Path("/content/drive/MyDrive/gridfm_opf")
    GENCO_ROOT = Path("/content/genco_results/opf_2seeds/experiments")
else:
    REPO = Path.cwd().resolve()
    while REPO != REPO.parent and not (
        REPO / "examples" / "config" / "HGNS_OPF_case118_tutorial.yaml"
    ).exists():
        REPO = REPO.parent
    DATA, LOG = REPO / "data", REPO / "mlruns"
    GENCO_ROOT = REPO / "data" / "genco_results" / "opf_2seeds" / "experiments"

CONFIG = REPO / "examples" / "config" / "HGNS_OPF_case118_tutorial.yaml"
assert CONFIG.is_file(), f"Config not found: {CONFIG}"
sys.path.insert(0, str(REPO / "examples" / "notebooks"))

import matplotlib.pyplot as plt
from mlflow.tracking import MlflowClient

from tutorial_opf_helpers import (
    compare_nb_test_genco_dc,
    compare_scratch_vs_finetune,
    latest_mlflow_run_id,
    link_paper_weights,
    plot_scratch_vs_finetune_val_curves,
    plot_test_gap_and_violations,
    plot_val_residual_boxplot,
    prepare_tutorial_opf_raw,
    solve_one_opf,
)

GENCO = link_paper_weights(REPO, GENCO_ROOT)
client = MlflowClient(LOG.resolve().as_uri())
print("REPO", REPO)
print("DATA", DATA)
print("LOG", LOG)


## What this tutorial is about

In this tutorial, we will learn to train a GENCO model using `gridfm-graphkit`. `gridfm-graphkit` is a library from the OpenGridFM LFE project that allows building, training and evaluating neural solvers for the grid.

While GENCO handles Optimal Power Flow (OPF), Power Flow (PF) and State Estimation (SE), we focus here on OPF.
This notebook is structured as follows:
1. What is AC OPF?
2. Data (This part will not be interactive)
3. The GENCO architecture
4. Training the model
5. Evaluating the model
6. Fine-tuning a model trained for a 2000-bus grid on a 118-bus grid.


# What is AC Optimal Power Flow?

Given a snapshot of the grid (loads, topology, generator costs and limits), AC-OPF finds the **cheapest generator dispatch** that still satisfies the load, power balance and operating limits.

The formulation below is a simplified version of the [`PowerModels` AC-OPF model](https://lanl-ansi.github.io/PowerModels.jl/stable/math-model/) (bus-injection form). Complex power is $S = P + jQ$.

**Variables**

- $S^g_k$ — complex power of generator $k$
- $V_i$ — complex voltage at bus $i$
- $S_{ij}$ — complex power flow on branch $(i,j)$ (both directions)

**Objective** — minimize total cost of generation (a quadratic function of active generations):

$$
\min \sum_{k \in G} c_{2k}\, P^g_k{}^2 + c_{1k}\, P^g_k + c_{0k}
\qquad\text{with } P^g_k = \mathrm{Re}(S^g_k)
$$

**Constraints**

1. Reference-bus angle: $\angle V_r = 0$
2. Generator limits: $\underline{S}^g_k \le S^g_k \le \overline{S}^g_k$
3. Voltage limits: $\underline{v}_i \le |V_i| \le \overline{v}_i$
4. Power balance (KCL) at each bus: generation minus load minus shunt equals outgoing branch flows
5. Branch flow from Ohm's law on the $\pi$-model (admittance $Y_{ij}$, tap $T_{ij}$)
6. Thermal limits: $|S_{ij}| \le \overline{s}_{ij}$
7. Angle-difference limits: $\underline{\theta}_{ij} \le \angle(V_i V_j^*) \le \overline{\theta}_{ij}$




## Why use GENCO?

Classical AC-OPF solvers (e.g. `IPOPT` via `PowerModels`) return a locally optimal $S^g, V, S_{ij}$. A neural OPF solver learns that mapping — **loads + costs + limits → dispatch and voltages** — much faster, with a small optimality gap.

The figure below is the qualitative picture: a classical AC solver is accurate but slow; a neural solver learns a mapping between solver inputs and outputs and is much faster at inference.

<img src="https://raw.githubusercontent.com/gridfm/gridfm-graphkit/lfe-tutorial/examples/notebooks/figures/classical_vs_neural.svg" alt="Classical AC solver vs neural solver" style="max-width:720px;width:100%"/>

A baseline for our neural solver is **DC-OPF**, a linear relaxation of OPF that can be solved much faster.

> DC-OPF drops reactive power, voltage magnitudes (fixed at 1 p.u.), and most AC physics. That is why it is fast, and why it misses $Q$, $|V|$, and many violations. The goal in this notebook is a solver that is **more accurate than DC and faster than AC**.


# Data

## Generating data with `gridfm-datakit`

In this tutorial, we will use data generated with `gridfm-datakit`, the data generation library part of the LFE OpenGridFM project.
`gridfm-datakit` uses Python and Julia. The installation of datakit is very easy and takes only a few minutes on a laptop. However, Colab makes Julia installation complicated, so data generation will be shown on the presenter's laptop only.


`gridfm-datakit` was checked against **real Hydro-Québec SCADA** snapshots: the same features (loads, voltages, dispatch) have comparable diversity to the synthetic HQ1200 set. That is the test that the generator is not just perturbing a single operating point.

<img src="https://raw.githubusercontent.com/gridfm/gridfm-graphkit/lfe-tutorial/examples/notebooks/figures/datakit_scada_entropy.png" alt="SCADA vs datakit feature entropy on HQ1200" style="max-width:360px;width:100%"/>

*Normalized feature entropy, GENCO paper: real SCADA (HQ1200) vs datakit on the same topology. Values near 1 are more diverse.*

This diversity comes from load-scenario generation **and** from something most OPF libraries omit: **varying generator costs**. Without cost variation, a neural OPF solver only ever sees one market.

<img src="https://raw.githubusercontent.com/gridfm/gridfm-datakit/refs/heads/main/docs/figs/comparison_table.png" alt="datakit vs other libraries" style="max-width:420px;width:100%"/>


**[demo from the presenter]**

## Loading pre-generated tutorial data

OpenGridFM makes many pre-generated datasets available on Hugging Face. Reusing this data allows: controlled benchmarking against other models that used the same data, and saving time: for large grids this data took several days on hundreds of CPUs to generate.

Here we use the `gridfm/opf_small_case118_ieee` dataset: about 200k solved OPF scenarios for the IEEE 118-bus system (the paper Small model was trained on 250k from the full Datakit release).

We will only use a subset of this — 1,000 scenarios — and pull them from Hugging Face directly.


In [ ]:
raw = prepare_tutorial_opf_raw(DATA, n_scenarios=1_000)
print("Hive raw at", raw)
print(
    "partitions",
    sorted(p.name for p in (raw / "bus_data.parquet").glob("scenario_partition=*"))[:5],
    "...",
)


The Hugging Face dataset contains statistical plots about the data:


![stats plot](https://huggingface.co/datasets/gridfm/opf_small_case118_ieee/resolve/main/stats_plot.png)

From left to right, top to bottom, we observe:
- Panel 1: All branches respect their thermal limits, as expected for feasible OPF solutions (confirmed by panel 10). A pronounced peak at 1 indicates that a non-negligible fraction of branches operate exactly at their thermal limits, suggesting that line congestion is frequently active in the generated OPF solutions.
- Panel 5: The mean active power balance equation residuals are very small ($< 10^{-9}$ MW) for the AC solver.
- Panel 6: for the DC solver, this quantity is 2.46 MW.
- Panel 8: There are between 176 and 186 branches in each scenario, indicating this data contains N-10 branch variations (up to 10 branches are disconnected compared to the base topology).
- Panel 9: There are between 46 and 54 generators. The data was also generated with up to 10 generators missing compared to the base case, but the data only shows up to 8 missing because more missing generators led to no convergence (not enough generation to satisfy load).
- Panel 10: The AC and DC solvers used to generate this data solved each scenario on average in 157 and 29 ms respectively. Note that this is CPU runtime, and would be smaller if we considered parallelism (in this time span many scenarios are solved because many other cores are working).


Feel free to explore this link to see how the data is organized: https://huggingface.co/datasets/gridfm/opf_small_case118_ieee/tree/main

And to read https://arxiv.org/abs/2512.14658 to learn more about the datakit.



## GENCO

**GENCO** (GEometric Neural Corrective Optimizer) is a heterogeneous graph transformer that maps a grid snapshot to an AC-OPF solution. It runs $N$ correction steps: each step updates bus and generator embeddings, decodes a candidate $(P_g, V, \theta)$, completes the state with a physics decoder, then feeds power-balance residuals back into the next step.

<img src="https://raw.githubusercontent.com/gridfm/gridfm-graphkit/lfe-tutorial/examples/notebooks/figures/genco_architecture.png" alt="GENCO architecture" style="max-width:900px;width:100%"/>

### Inputs

The grid is a heterogeneous graph: **bus** nodes, **generator** nodes, directed **branch** edges, and featureless bus–generator links. Unknown OPF variables are zeroed; loads, limits, costs, and topology stay visible.

| Node / edge | What is fed in | YAML |
| --- | --- | --- |
| Bus | $P_d$, $Q_d$, $Q_g$, $V$, $\theta$, PQ/PV/REF flags, $V$ limits, $Q_g$ limits, shunts | `model.input_bus_dim: 15` |
| Generator | $P_g$, $P_g$ limits, quadratic cost $(c_0, c_1, c_2)$ | `model.input_gen_dim: 6` |
| Branch | admittance, tap, angle limits, thermal rating | `model.edge_dim: 10` |

Width and depth of the latent corrections are `model.hidden_size: 12`, `model.num_layers: 12` (one HT layer per correction step), and `model.attention_head: 8`.

### Outputs

The last correction step is the OPF prediction $x_N$:

- **Buses** — $V$ and $\theta$ (`model.output_bus_dim: 2`). $V$ is mapped into $[\underline{V}, \overline{V}]$ with a sigmoid (the **Sigmoid Bound** / OPF decoder in the figure).
- **Generators** — $P_g$ (`model.output_gen_dim: 1`), likewise boxed into $[\underline{P}_g, \overline{P}_g]$.
- **Completed state** — branch flows from Ohm's law; $Q_g$ recovered from reactive power balance (not a free decoder output).


### Loss

`training.losses`, `training.loss_weights`, and `training.loss_args` are aligned lists. The objective is the weighted sum

$$
{L}
=
0.1\,{L}_{\mathrm{phys}}
+
0.1\,{L}_{P_g}
+
0.75\,{L}_{V,\theta}
+
0.001\,{L}_{Q_g}.
$$

| Weight | YAML name | Role |
| --- | --- | --- |
| **0.75** | `MaskedBusMSE` | MSE of predicted vs `IPOPT` $V$ and $\theta$ (largest term). |
| **0.1** | `MaskedGenMSE` | MSE of predicted vs `IPOPT` $P_g$. |
| **0.1** | `LayeredWeightedPhysics` | Weighted sum of the mean power-balance residual at every correction step. Inside this term, `loss_args.base_weight: 0.5` gives geometrically **larger weight to later steps** (same $\lambda^{N-i}$ idea as in the figure) |
| **0.001** | `QgViolationPenalty` | Penalty when recovered $Q_g$ leaves $[\underline{Q}_g, \overline{Q}_g]$. |

So the YAML is telling GENCO: match the `IPOPT` voltages most strongly, pull generator $P_g$ and physics residuals in more weakly, and keep a small $Q_g$ bound penalty.


# Training the model

We now train a tiny GENCO model (hidden size 12, ~1.3M parameters) for OPF on the IEEE 118-bus system.


We will use the same CLI you would run in a terminal.

``bash
gridfm_graphkit train --config <config.yaml> --data_path <data> --log_dir <logs>
``

That command loads the YAML, builds a Datamodule with the data we have loaded, constructs the OPF task, and runs `trainer.fit`.


The tutorial uses `examples/config/HGNS_OPF_case118_tutorial.yaml`. There are other configs (for other grids and tasks) in `examples/config/`. The next cell prints that file. What each field does:

**`task`**
- `task_name: OptimalPowerFlow` — predict voltages and generator dispatch.

**`data`**
- `scenarios: [1000]` — use the first 1,000 scenarios of that network.
- `test_ratio` / `val_ratio` — 10% test, 10% val, remaining ~80% train.
- `mask_value: 0.0` — fill value for masked features.

**`model`** (GENCO)
- `hidden_size`, `num_layers`, `attention_head` — width and depth (12 layers, hidden 12, 8 heads).

**`optimizer`**
- AdamW at `learning_rate: 0.0005`, `betas: [0.9, 0.999]`.
- `ReduceLROnPlateau` multiplies LR by `0.7` after `patience: 5` epochs of no improvement (`mode: min`).

**`training`**
- `batch_size: 64`, `epochs: 20`.
- `accelerator` / `devices` / `strategy: auto` — `Lightning` device selection (GPU/MPS/CPU).
- Losses (order matches `loss_weights` and `loss_args`):
  - `LayeredWeightedPhysics` (0.1) — power-balance residuals; `base_weight: 0.5` scales deeper layers.
  - `MaskedGenMSE` (0.1) — match \(P_g\) on generators.
  - `MaskedBusMSE` (0.75) — match bus \(V, \theta\).
  - `QgViolationPenalty` (0.001) — penalize reactive generation outside limits.


In [ ]:
print(CONFIG.read_text())


We can then run the following cell to start training for **20 epochs**.


In [ ]:
!gridfm_graphkit train \
  --config {CONFIG} \
  --data_path {DATA} \
  --log_dir {LOG} \
  --exp_name tutorial_opf \
  --log_every_n_steps 10 \
  --compute_dc_ac_metrics \
  --num_workers 2 \
  --mp_context spawn


After training, we can check how the training and validation losses look:


In [ ]:
rid = client.search_runs(
    [e.experiment_id for e in client.search_experiments()],
    order_by=["start_time DESC"],
    max_results=1,
)[0].info.run_id


def hist(name):
    try:
        h = sorted(client.get_metric_history(rid, name), key=lambda m: m.step)
    except Exception:
        return [], []
    return [m.step for m in h], [m.value for m in h]


fig, ax = plt.subplots(figsize=(8, 4))
for name, fmt in (
    ("Training Loss", "o-"),
    ("Training Loss_step", "o-"),
    ("Validation loss", "s-"),
):
    s, v = hist(name)
    if v:
        ax.plot(s, v, fmt, label=name)
ax.set_xlabel("step")
ax.set_ylabel("loss")
ax.legend()
plt.show()


The validation loss can be below the training loss: that is because the training loss is an average of the loss of each batch during training, while the validation loss is obtained after running the last update of the model for a given epoch on the validation set.


After each correction step, GENCO decodes $(P_g, V, \theta)$, completes flows from Ohm's law, and checks **nodal power balance**.

The curve below is the last-step validation residual (`Validation layer_11_residual`): the mean power-balance residual of the predictions, in per-unit.

A **downward** trend means voltages and dispatch are getting closer to the power-flow equations (the OPF equality constraints). The physics term in the YAML (`LayeredWeightedPhysics`) is what pushes this down. The curve is computed on **held-out load scenarios** of the same base grid (topology, load, and parameter changes), not a different network.


In [ ]:
s, v = hist("Validation layer_11_residual")
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(s, v, "o-")
ax.set_xlabel("training step")
ax.set_ylabel("power-balance residual (p.u.)")
ax.set_title("Validation set: last-step power-balance residual")
plt.show()


GENCO is a **corrective** solver: each step proposes a candidate OPF state, measures how far it is from power balance, and feeds that residual into the next step. If that design works, residuals on held-out scenarios should **shrink with depth** — later steps closer to power balance than early ones, last step the tightest.

The box plot below checks that on the **validation set** after training: one box per correction step, the spread of per-bus residuals.


In [ ]:
plot_val_residual_boxplot(CONFIG, DATA, client, rid);


A downward shift across correction steps confirms the iterative physics feedback is actually refining the solution, not just fitting `IPOPT` voltages in a single shot.

As a control, the same plot for an **untrained** model (orange) against the **trained** model (blue). If residuals fell with depth even at random initialization, the architecture alone would be doing the work — we want the drop to appear only after training.


In [ ]:
plot_val_residual_boxplot(CONFIG, DATA, client, rid, overlay=True);


Untrained residuals stay large at every step; the trained model is the one that tightens power balance with depth.


We now look at **test-set** results: the held-out 10% of this notebook run. The table compares the Tiny GENCO we just trained to DC-OPF on that split. Both columns are read from `MLflow` `artifacts/test/` (`*_metrics.csv` for GENCO, `*_opf_ac_dc_metrics.csv` for DC-OPF), as the test was already run automatically at the end of training.


In [ ]:
compare_nb_test_genco_dc(client, rid)


These test numbers are **too weak** — Tiny loses to DC-OPF here because we did not train long enough (and not on enough data). This notebook run is **20 epochs** on **1,000** scenarios with `hidden_size: 12` (~1.3M parameters).

The IEEE 118 **Small** model in the paper was trained for **200 epochs** on the full Datakit **250,000** scenarios with `hidden_size: 24` (~5M parameters). The loss, optimizer, and LR schedule (`ReduceLROnPlateau`, factor 0.7, patience 5) are the same; what changed is the budget: more data, more epochs, and a wider model. The next cells load that paper checkpoint.


# Evaluate GENCO Small (paper checkpoint)

The next cell loads one of the models used for the GENCO paper results on IEEE 118 and runs `gridfm_graphkit evaluate` with that model's weights.


In [ ]:
CONFIG_SMALL = REPO / "examples" / "config" / "HGNS_OPF_case118_small.yaml"
SMALL_RUN = GENCO / "465594abc8eb4c96b908363afc84a0e1"
MODEL_PATH = SMALL_RUN / "artifacts" / "model" / "best_model_state_dict.pt"
NORM_PATH = SMALL_RUN / "artifacts" / "stats" / "normalizer_stats.pt"
assert MODEL_PATH.is_file(), f"Missing checkpoint: {MODEL_PATH}"
assert NORM_PATH.is_file(), f"Missing normalizer: {NORM_PATH}"
# Paper YAMLs look for data/case118_ieee; the download writes nb_opf_case118.
_src, _alias = DATA / "nb_opf_case118", DATA / "case118_ieee"
if not _alias.exists() and _src.is_dir():
    _alias.symlink_to(_src.resolve(), target_is_directory=True)

!gridfm_graphkit evaluate \
  --config {CONFIG_SMALL} \
  --data_path {DATA} \
  --log_dir {LOG} \
  --exp_name tutorial_opf_eval_small \
  --model_path {MODEL_PATH} \
  --normalizer_stats {NORM_PATH} \
  --compute_dc_ac_metrics \
  --num_workers 2 \
  --mp_context spawn


After running the evaluation, we can now look at the same metrics as before, but for this model:


In [ ]:
eval_exp = [e for e in client.search_experiments() if e.name == "tutorial_opf_eval_small"]
eval_rid = client.search_runs(
    [eval_exp[0].experiment_id],
    order_by=["start_time DESC"],
    max_results=1,
)[0].info.run_id
compare_nb_test_genco_dc(client, eval_rid)


On this 1,000-scenario IEEE 118 split, the paper **Small** checkpoint beats DC-OPF on every comparable number: optimality gap **1.16% vs 4.82%**, active residual **0.97 MW vs 2.54 MW**, and smaller thermal violations. DC-OPF has no reactive residual or $Q_g$ violation to report (it does not model $Q$). That is the paper-scale model — not the Tiny run we trained for 20 epochs above.


We overlay GENCO vs DC-OPF distributions of optimality gap and constraint violations (mean over buses or branches in each scenario).


In [ ]:
plot_test_gap_and_violations(CONFIG_SMALL, DATA, MODEL_PATH, NORM_PATH)


# Solving OPF with GENCO

Inference is one forward pass. The next cell loads the paper **Small** checkpoint, takes the **first validation scenario** on IEEE 118, and prints:

1. feasibility and optimality as mean violation percentages (value − bound) / bound, ignoring zero bounds; power-balance uses net injection (load) as the bound; total active imbalance is the signed sum of bus $P$ residuals as a percentage of total active injections
2. generator dispatch next to `IPOPT` and DC-OPF
3. bus active residuals


In [ ]:
solve_one_opf(
    CONFIG_SMALL,
    DATA,
    model_path=MODEL_PATH,
    normalizer_stats=NORM_PATH,
)


We see the following:

-


# Fine-tuning an OPF model for IEEE 2000 to IEEE 118


Training from scratch is expensive. Our Tiny run (20 epochs, 1,000 scenarios) did **not** beat DC-OPF. The paper Small checkpoint used here was trained for **200 epochs** on the full **250,000**-scenario Datakit set.

That much data is often unavailable for a real TSO grid. So the useful question is: can we **reuse** a model trained on another grid and adapt it with a small slice?

This last experiment does that on IEEE 118:

- **Scratch** — train Small (`hidden_size: 24`) from random weights for **20 epochs** on the same 1,000-scenario slice
- **Fine-tune** — start from the paper **case2000 Small** weights and adapt them to 118 for the same 20 epochs

If transfer works, fine-tune should beat scratch (and move toward DC or better) with the same budget.


In [ ]:
CONFIG_FEW_SMALL = REPO / "examples" / "config" / "HGNS_OPF_case118_fewshot_small.yaml"
PRETRAINED_2K_SMALL = (
    GENCO / "f227514415974447befc64513a758432" / "artifacts" / "model" / "best_model_state_dict.pt"
)
assert CONFIG_FEW_SMALL.is_file(), f"Missing config: {CONFIG_FEW_SMALL}"
assert PRETRAINED_2K_SMALL.is_file(), f"Missing case2000 Small checkpoint: {PRETRAINED_2K_SMALL}"
_src, _alias = DATA / "nb_opf_case118", DATA / "case118_ieee"
if not _alias.exists() and _src.is_dir():
    _alias.symlink_to(_src.resolve(), target_is_directory=True)

!gridfm_graphkit train \
  --config {CONFIG_FEW_SMALL} \
  --data_path {DATA} \
  --log_dir {LOG} \
  --exp_name tutorial_opf_scratch_case2000_small20 \
  --log_every_n_steps 10 \
  --compute_dc_ac_metrics \
  --num_workers 2 \
  --mp_context spawn

!gridfm_graphkit finetune \
  --config {CONFIG_FEW_SMALL} \
  --data_path {DATA} \
  --log_dir {LOG} \
  --exp_name tutorial_opf_ft_case2000_small \
  --model_path {PRETRAINED_2K_SMALL} \
  --log_every_n_steps 10 \
  --compute_dc_ac_metrics \
  --num_workers 2 \
  --mp_context spawn


Validation curves for the two 20-epoch runs: **scratch** vs **case2000 → 118 fine-tune**. The test-set table against DC-OPF is in the next cell (same metrics as the paper Small evaluation).


In [ ]:
plot_scratch_vs_finetune_val_curves(
    client,
    latest_mlflow_run_id(client, "tutorial_opf_scratch_case2000_small20"),
    latest_mlflow_run_id(client, "tutorial_opf_ft_case2000_small"),
    metrics=("Validation loss",),
);


We can see that the fine-tuned model has a much lower validation loss than the model trained from scratch, indicating that pretraining on another grid (case2000) transfers useful structure to IEEE 118: with the same 20-epoch budget, the adapted model learns the 118-bus OPF mapping faster than random initialization.


In [ ]:
compare_scratch_vs_finetune(
    client,
    latest_mlflow_run_id(client, "tutorial_opf_scratch_case2000_small20"),
    latest_mlflow_run_id(client, "tutorial_opf_ft_case2000_small"),
)


Fine-tuning from case2000 beats training from scratch on the same small budget: lower optimality gap, smaller power-balance residuals, and much tighter generator $Q$ limits. It moves toward DC-OPF on cost and active mismatch, but does not overtake it here. DC-OPF has empty reactive columns because it does not model $Q$. Thermal violations stay small for all three and do not show a stable ranking from run to run.


# Conclusion

We trained a GENCO OPF model in **`gridfm-graphkit`** on **`gridfm-datakit`** scenarios. A short Tiny run shows the corrective residual drop but does not beat DC-OPF. A larger model trained longer (the paper Small checkpoint) does. Fine-tuning a model from another grid (case2000 → 118) also reaches a useful solution faster than training from scratch on the same small budget.

**Resources**

- [`gridfm-graphkit`](https://github.com/gridfm/gridfm-graphkit)
- [`gridfm-datakit`](https://github.com/gridfm/gridfm-datakit)
- GENCO paper: [arXiv:2608.09921](https://arxiv.org/abs/2608.09921)
- datakit paper: [arXiv:2512.14658](https://arxiv.org/abs/2512.14658)

Questions: Alban Puech — Alban.Puech2@ibm.com
